# Qwen-Image-2.1 (G4 / H100 / A100 · vLLM-Omni T2I, 1-4 Ref RGBA I2I & Drive I/O)

Serves `Qwen/Qwen-Image-2.1` (7.1B single-stream DiT with cross-step prefix KV cache + Qwen3-VL-8B vision-language encoder + 16x 4-channel RGBA VAE) on a Colab G4 GPU by default (also compatible with `--gpu H100` and `--gpu A100 --quantization fp8-all`) via `vllm serve --omni` with fail-closed Bearer token auth, CORS preflight support, OpenAI-compatible `/v1/images/generations`, `/v1/images/edits`, `/api/ct/edit`, and `/v1/chat/completions`, 1–4 `<image1>..<image4>` reference images, RGBA transparency, direct Google Drive reference image reading & output persistence, and LoRA.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "qwen-image")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8091"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8091"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- install vLLM + vLLM-Omni (Qwen-Image-2.1) & stage checkpoint on local NVMe SSD
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

QI_SETUP_OK = False
MODEL_REPO = os.environ.get('CT_QWEN_IMAGE_MODEL', 'Qwen/Qwen-Image-2.1').strip() or 'Qwen/Qwen-Image-2.1'
MODEL_ALIAS = os.environ.get('CT_QWEN_IMAGE_ALIAS', 'qwen-image-2.1').strip() or 'qwen-image-2.1'
QUANTIZATION = os.environ.get('CT_QWEN_IMAGE_QUANTIZATION', 'none').strip().lower() or 'none'
PREFIX_KV_DTYPE = os.environ.get('CT_QWEN_IMAGE_PREFIX_KV_DTYPE', 'auto').strip().lower() or 'auto'
STEP_EXECUTION = os.environ.get('CT_QWEN_IMAGE_STEP_EXECUTION', '0').strip() in ('1', 'true', 'True', 'yes')
VAE_TILING = os.environ.get('CT_QWEN_IMAGE_VAE_TILING', '1').strip() not in ('0', 'false', 'False', 'no')
_raw_mns = os.environ.get('CT_QWEN_IMAGE_MAX_NUM_SEQS', '').strip()
MAX_NUM_SEQS = int(_raw_mns) if _raw_mns else (8 if STEP_EXECUTION else 4)
LORA_PATH_SPEC = os.environ.get('CT_QWEN_IMAGE_LORA_PATH', '').strip()
LORA_SCALE = os.environ.get('CT_QWEN_IMAGE_LORA_SCALE', '1.0').strip() or '1.0'
VLLM_VERSION = os.environ.get('CT_QWEN_IMAGE_VLLM_VERSION', '0.31.0').strip() or '0.31.0'
VLLM_OMNI_REF = os.environ.get('CT_VLLM_OMNI_REF', '389a02a149ced03a9550368565571fff9d1d99f3').strip() or '389a02a149ced03a9550368565571fff9d1d99f3'

QI_WORK = Path('/content/qwen-image')
VENV_DIR = QI_WORK / 'venv'
MODELS_ROOT = QI_WORK / 'model'
MODEL_DIR = MODELS_ROOT / MODEL_REPO.split('/')[-1]
LOCAL_LORA_DIR = QI_WORK / 'loras'
DRIVE_INPUTS_DIR = Path(DATA_DIR) / 'inputs'
DRIVE_OUTPUTS_DIR = Path(DATA_DIR) / 'outputs'
DRIVE_LORAS_DIR = Path(DATA_DIR) / 'loras'

FLASHINFER_DRIVE_DIR = Path(DATA_DIR) / 'flashinfer-cache'
FLASHINFER_CACHE_DIR = QI_WORK / 'flashinfer-cache'
TRITON_DRIVE_DIR = Path(DATA_DIR) / 'triton-cache'
TRITON_CACHE_DIR = QI_WORK / 'triton-cache'
VLLM_DRIVE_DIR = Path(DATA_DIR) / 'vllm-cache'
VLLM_CACHE_DIR = QI_WORK / 'vllm-cache'

for _d in (
    QI_WORK, MODELS_ROOT, LOCAL_LORA_DIR,
    DRIVE_INPUTS_DIR, DRIVE_OUTPUTS_DIR, DRIVE_LORAS_DIR,
    FLASHINFER_CACHE_DIR, TRITON_CACHE_DIR, VLLM_CACHE_DIR,
):
    _d.mkdir(parents=True, exist_ok=True)

def qilog(msg):
    print(time.strftime('[%F %T] [qwen-image] ') + msg, flush=True)

(QI_WORK / 'colab_cache.py').write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
if str(QI_WORK) not in sys.path:
    sys.path.insert(0, str(QI_WORK))
from colab_cache import restore_cache, save_cache

try:
    _gpu_arch = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
except Exception:
    _gpu_arch = "90"
CACHE_KEY = f"vllm-{VLLM_VERSION}-qwenimg21-{VLLM_OMNI_REF[:8]}-sm{_gpu_arch.replace('.', '')}"
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_PAIRS = (
    (FLASHINFER_CACHE_DIR, FLASHINFER_DRIVE_DIR),
    (TRITON_CACHE_DIR, TRITON_DRIVE_DIR),
    (VLLM_CACHE_DIR, VLLM_DRIVE_DIR),
)
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        restore_cache(_local_c, _legacy_c, CACHE_ARCHIVES, CACHE_KEY, qilog)
    except (OSError, ValueError) as _err:
        qilog(f"Cache restore warning ({_local_c.name}): {_err}; continuing with local SSD cache")

def run_logged(cmd, log_path, env=None, cwd=None, check=True):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True)
    if check and res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-45:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:4])}\n{tail}")
    return res

qilog(
    f"GPU detected: {ct_gpu_stat('name,memory.total,compute_cap')} | "
    f"repo={MODEL_REPO} quant={QUANTIZATION} step_exec={STEP_EXECUTION} max_seqs={MAX_NUM_SEQS}"
)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

def prepare_env():
    env_log = QI_WORK / "env-setup.log"
    marker = VENV_DIR / f".ct_qwen_img_env_{VLLM_VERSION}_{VLLM_OMNI_REF[:10]}.ok"
    vllm_bin = VENV_DIR / "bin" / "vllm"
    py_bin = VENV_DIR / "bin" / "python"
    if marker.is_file() and vllm_bin.is_file() and py_bin.is_file():
        qilog("Reusing existing vLLM-Omni (Qwen-Image-2.1) virtualenv")
        return VENV_DIR

    qilog(f"Installing vllm=={VLLM_VERSION} + vllm-omni@{VLLM_OMNI_REF[:10]} (log: {env_log})...")
    uv_bin = shutil.which("uv") or "uv"
    run_logged([uv_bin, "venv", "--python", "3.12", str(VENV_DIR), "--clear"], env_log)
    run_logged(
        [
            uv_bin, "pip", "install", "--python", str(py_bin),
            f"vllm=={VLLM_VERSION}",
            f"vllm-omni @ git+https://github.com/vllm-project/vllm-omni.git@{VLLM_OMNI_REF}",
            "pillow", "huggingface_hub", "hf_xet",
        ],
        env_log,
    )
    marker.write_text("ok\n", encoding="utf-8")
    qilog("vLLM-Omni (Qwen-Image-2.1) virtualenv ready")
    return VENV_DIR

def _cleanup_hf_temp():
    for _p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(_p, ignore_errors=True)
    for _mp in MODELS_ROOT.glob("*/.cache"):
        shutil.rmtree(_mp, ignore_errors=True)

def prepare_model():
    dl_log = QI_WORK / "model-download.log"
    ready_stamp = MODEL_DIR / ".ct_model_ready"
    if (
        ready_stamp.is_file()
        and (MODEL_DIR / "model_index.json").is_file()
        and (MODEL_DIR / "vae" / "diffusion_pytorch_model.safetensors").is_file()
    ):
        qilog(f"Reusing staged local SSD checkpoint at {MODEL_DIR}")
        return MODEL_DIR

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
    _cleanup_hf_temp()
    free_gb = shutil.disk_usage("/content").free / 1e9
    qilog(f"Staging {MODEL_REPO} (~33.1 GB) onto local SSD (free SSD: {free_gb:.1f} GB; log: {dl_log})...")
    dl_env = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    dl_env.pop("HF_XET_HIGH_PERFORMANCE", None)

    dl_script = r'''
import os, shutil, sys
from pathlib import Path
from huggingface_hub import snapshot_download

model_repo, model_dir_s = sys.argv[1:3]
model_dir = Path(model_dir_s)
token = os.environ.get("HF_TOKEN") or None
model_dir.mkdir(parents=True, exist_ok=True)
print(f"[dl] Downloading {model_repo} -> {model_dir}", flush=True)
snapshot_download(
    repo_id=model_repo,
    local_dir=str(model_dir),
    token=token,
    max_workers=4,
    ignore_patterns=["*.md", ".gitattributes"],
)
for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
    shutil.rmtree(p, ignore_errors=True)
shutil.rmtree(model_dir / ".cache", ignore_errors=True)
(model_dir / ".ct_model_ready").write_text(f"{model_repo}\n", encoding="utf-8")
'''
    try:
        run_logged([sys.executable, "-c", dl_script, MODEL_REPO, str(MODEL_DIR)], dl_log, env=dl_env)
    finally:
        _cleanup_hf_temp()
    qilog(f"Checkpoint staged on local SSD (free SSD: {shutil.disk_usage('/content').free / 1e9:.1f} GB)")
    return MODEL_DIR

with ThreadPoolExecutor(max_workers=2) as _pool:
    _f_env = _pool.submit(prepare_env)
    _f_model = _pool.submit(prepare_model)
    while not (_f_env.done() and _f_model.done()):
        for _fut in (_f_env, _f_model):
            if _fut.done() and _fut.exception():
                raise _fut.exception()
        qilog(
            f"Setup heartbeat | env={'done' if _f_env.done() else 'running'} "
            f"model={'done' if _f_model.done() else 'running'} | "
            f"SSD free={shutil.disk_usage('/content').free / 1e9:.1f} GB"
        )
        time.sleep(20)
    _f_env.result()
    _f_model.result()
QI_SETUP_OK = True
qilog("Environment and local SSD model staging complete")


In [ ]:
# @title 2. Service -- start vLLM-Omni Qwen-Image-2.1 server on loopback
if not globals().get('QI_SETUP_OK'):
    raise RuntimeError('Qwen-Image-2.1 setup cell failed before completing model staging; check setup traceback above.')
(QI_WORK / 'colab_asgi_common.py').write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    try:\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except OSError as os_err:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {os_err}") from os_err\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(QI_WORK / 'ct_qwen_image_middleware.py').write_text('"""Authenticated ASGI middleware, Google Drive image/LoRA staging, and RGBA/edit routing for Qwen-Image-2.1."""\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nfrom email.parser import BytesParser\nfrom email.policy import default as email_default_policy\nimport glob as glob_mod\nimport hashlib\nfrom http import HTTPStatus\nimport io\nimport json\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport shutil\nimport time\nfrom typing import Any\nimport urllib.error\nfrom urllib.parse import parse_qs\nimport urllib.request\n\nfrom colab_asgi_common import (\n    DEFAULT_SKIP_DIRS,\n    allowed_drive_roots,\n    rel_to_mydrive,\n    resolve_drive_path,\n    sanitize_header_param,\n    tail_service_log,\n    verify_bearer_auth,\n)\n\ntry:\n    from starlette.datastructures import Headers\n    from starlette.responses import JSONResponse, Response\n    from starlette.types import ASGIApp, Message, Receive, Scope, Send\nexcept ImportError:  # pragma: no cover - local unit-test fallback without starlette\n    ASGIApp = Receive = Scope = Send = Message = Any  # type: ignore[assignment,misc]\n\n    class Headers:  # type: ignore[no-redef]\n        def __init__(self, scope: dict[str, Any]) -> None:\n            self._d = {k.decode("latin1").lower(): v.decode("latin1") for k, v in scope.get("headers", [])}\n\n        def get(self, key: str, default: str = "") -> str:\n            return self._d.get(key.lower(), default)\n\n    class Response:  # type: ignore[no-redef]\n        def __init__(self, content: bytes = b"", status_code: int = 200, media_type: str | None = None) -> None:\n            self.body = content\n            self.status_code = int(status_code)\n            self.media_type = media_type\n\n        async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n            hdrs = [(b"content-length", str(len(self.body)).encode("ascii"))]\n            if self.media_type:\n                hdrs.append((b"content-type", self.media_type.encode("ascii")))\n            await send({"type": "http.response.start", "status": self.status_code, "headers": hdrs})\n            await send({"type": "http.response.body", "body": self.body, "more_body": False})\n\n    class JSONResponse(Response):  # type: ignore[no-redef]\n        def __init__(self, content: Any, status_code: int = 200) -> None:\n            super().__init__(json.dumps(content).encode("utf-8"), status_code=status_code, media_type="application/json")\n\nIMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tiff", ".tif")\nLORA_EXTS = (".safetensors", ".pt", ".bin")\nMIME_BY_EXT = {\n    ".png": "image/png",\n    ".jpg": "image/jpeg",\n    ".jpeg": "image/jpeg",\n    ".webp": "image/webp",\n    ".bmp": "image/bmp",\n    ".gif": "image/gif",\n    ".tiff": "image/tiff",\n    ".tif": "image/tiff",\n}\nEXT_BY_MIME = {\n    "image/png": "png",\n    "image/jpeg": "jpg",\n    "image/webp": "webp",\n    "image/gif": "gif",\n}\n\n# Qwen-Image-2.1 supports 1-4 reference images (<image1> .. <image4>); a 5th is rejected with 400.\nMAX_REFERENCE_IMAGES = 4\nDEFAULT_STEPS = 40\nDEFAULT_TRUE_CFG_NO_NEG = 1.0\nDEFAULT_TRUE_CFG_WITH_NEG = 4.0\nDIM_ALIGNMENT = 32\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nSERVICE_PORT = int((os.environ.get("CT_SERVICE_PORT") or "").strip() or "8091")\n\nWORK_DIR = Path(os.environ.get("CT_QWEN_IMAGE_WORK_DIR", "/content/qwen-image"))\nMODEL_DIR = Path(os.environ.get("CT_QWEN_IMAGE_MODEL_DIR", str(WORK_DIR / "model" / "Qwen-Image-2.1")))\nLOCAL_LORA_DIR = Path(os.environ.get("CT_QWEN_IMAGE_LOCAL_LORA_DIR", str(WORK_DIR / "loras")))\n\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR", "/content/drive/MyDrive/colab-llm/qwen-image"))\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = Path("/content/drive/MyDrive")\nDRIVE_INPUTS_DIR = DATA_DIR / "inputs"\nDRIVE_OUTPUTS_DIR = DATA_DIR / "outputs"\nDRIVE_LORAS_DIR = DATA_DIR / "loras"\n\nMODEL_REPO = os.environ.get("CT_QWEN_IMAGE_MODEL", "Qwen/Qwen-Image-2.1")\nMODEL_ALIAS = os.environ.get("CT_QWEN_IMAGE_ALIAS", "qwen-image-2.1")\nQUANTIZATION = os.environ.get("CT_QWEN_IMAGE_QUANTIZATION", "none").strip().lower() or "none"\nPREFIX_KV_DTYPE = os.environ.get("CT_QWEN_IMAGE_PREFIX_KV_DTYPE", "auto").strip().lower() or "auto"\nSTEP_EXECUTION = os.environ.get("CT_QWEN_IMAGE_STEP_EXECUTION", "0").strip() in ("1", "true", "True", "yes")\nVAE_TILING = os.environ.get("CT_QWEN_IMAGE_VAE_TILING", "1").strip() not in ("0", "false", "False", "no")\n_RAW_MAX_NUM_SEQS = (os.environ.get("CT_QWEN_IMAGE_MAX_NUM_SEQS") or "").strip()\nMAX_NUM_SEQS = int(_RAW_MAX_NUM_SEQS) if _RAW_MAX_NUM_SEQS else (8 if STEP_EXECUTION else 4)\nPRELOADED_LORA = os.environ.get("CT_QWEN_IMAGE_PRELOADED_LORA", "").strip()\n\nCORS_HEADERS: tuple[tuple[bytes, bytes], ...] = (\n    (b"access-control-allow-origin", b"*"),\n    (b"access-control-allow-methods", b"GET, POST, OPTIONS"),\n    (\n        b"access-control-allow-headers",\n        b"Authorization, Content-Type, X-API-Key, X-Save-Drive, X-Drive-Only",\n    ),\n    (b"access-control-expose-headers", b"X-Saved-Drive-Path"),\n    (b"access-control-max-age", b"86400"),\n)\n\n\ndef _verify_auth(headers: Any) -> bool:\n    return verify_bearer_auth(headers, API_TOKEN)\n\n\ndef detect_image_mime(data: bytes, fallback_name: str = "") -> str:\n    if data.startswith(b"\\x89PNG\\r\\n\\x1a\\n"):\n        return "image/png"\n    if data.startswith(b"\\xff\\xd8\\xff"):\n        return "image/jpeg"\n    if len(data) >= 12 and data[:4] == b"RIFF" and data[8:12] == b"WEBP":\n        return "image/webp"\n    if data.startswith((b"GIF87a", b"GIF89a")):\n        return "image/gif"\n    if data.startswith(b"BM"):\n        return "image/bmp"\n    if fallback_name:\n        ext = Path(fallback_name).suffix.lower()\n        if ext in MIME_BY_EXT:\n            return MIME_BY_EXT[ext]\n        guessed, _ = mimetypes.guess_type(fallback_name)\n        if guessed and guessed.startswith("image/"):\n            return guessed\n    return "image/png"\n\n\ndef image_path_to_data_url(image_path: Path) -> str:\n    """Read an image file and encode it as a base64 data URI."""\n    if not image_path.is_file():\n        raise FileNotFoundError(f"Image file not found on Google Drive: {image_path}")\n    raw = image_path.read_bytes()\n    if not raw:\n        raise ValueError(f"Image file is empty: {image_path}")\n    mime = detect_image_mime(raw, image_path.name)\n    b64 = base64.b64encode(raw).decode("ascii")\n    return f"data:{mime};base64,{b64}"\n\n\ndef parse_size_spec(\n    size_str: str,\n    *,\n    allow_auto: bool = False,\n    align_to_32: bool = True,\n) -> tuple[int | None, int | None]:\n    """Parse \'WIDTHxHEIGHT\' (or \'auto\' when allow_auto=True), aligning to 32px."""\n    s = str(size_str or "").strip()\n    if allow_auto and s.lower() in ("", "auto"):\n        return None, None\n    m = re.fullmatch(r"(\\d+)\\s*[xX:]\\s*(\\d+)", s)\n    if not m:\n        expected = "WIDTHxHEIGHT or \'auto\'" if allow_auto else "WIDTHxHEIGHT (e.g. 1024x1024)"\n        raise ValueError(f"Invalid size \'{size_str}\': expected {expected}")\n    w, h = int(m.group(1)), int(m.group(2))\n    if w < 64 or h < 64 or w > 4096 or h > 4096:\n        raise ValueError(f"Invalid size {w}x{h}: width and height must be between 64 and 4096")\n    if align_to_32:\n        w = max(64, (w // DIM_ALIGNMENT) * DIM_ALIGNMENT)\n        h = max(64, (h // DIM_ALIGNMENT) * DIM_ALIGNMENT)\n    elif w % DIM_ALIGNMENT != 0 or h % DIM_ALIGNMENT != 0:\n        raise ValueError(f"Invalid size {w}x{h}: width and height must be multiples of {DIM_ALIGNMENT}")\n    return w, h\n\n\ndef _maybe_convert_color_format(raw_bytes: bytes, color_format: str | None) -> bytes:\n    """Convert 4-channel RGBA PNG bytes to 3-channel RGB PNG only when color_format==\'RGB\'."""\n    norm = (color_format or "").strip().upper()\n    if norm != "RGB":\n        return raw_bytes\n    try:\n        from PIL import Image\n\n        with Image.open(io.BytesIO(raw_bytes)) as im:\n            if im.mode == "RGB":\n                return raw_bytes\n            if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):\n                alpha = im.convert("RGBA")\n                flat = Image.new("RGB", alpha.size, (255, 255, 255))\n                flat.paste(alpha, mask=alpha.getchannel("A"))\n            else:\n                flat = im.convert("RGB")\n            buf = io.BytesIO()\n            flat.save(buf, format="PNG")\n            return buf.getvalue()\n    except Exception:\n        return raw_bytes\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef list_drive_files_by_kind(\n    spec: str,\n    *,\n    kind: str = "images",\n    data_dir: Path = DATA_DIR,\n    recursive: bool = True,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List image or LoRA files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    norm_kind = (kind or "images").strip().lower()\n    if norm_kind in ("lora", "loras"):\n        default_subdir = "loras"\n        allowed_exts = LORA_EXTS\n        include_peft_dirs = True\n    elif norm_kind in ("output", "outputs"):\n        default_subdir = "outputs"\n        allowed_exts = IMAGE_EXTS\n        include_peft_dirs = False\n    elif norm_kind == "all":\n        default_subdir = None\n        allowed_exts = IMAGE_EXTS + LORA_EXTS\n        include_peft_dirs = True\n    else:\n        default_subdir = "inputs" if not spec and (data_dir / "inputs").is_dir() else None\n        allowed_exts = IMAGE_EXTS\n        include_peft_dirs = False\n\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[tuple[Path, str]] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots) or str(hp) in seen:\n                    continue\n                if hp.is_file() and hp.suffix.lower() in allowed_exts:\n                    seen.add(str(hp))\n                    matched_paths.append((hp, "lora" if hp.suffix.lower() in LORA_EXTS else "image"))\n                elif include_peft_dirs and hp.is_dir() and (hp / "adapter_config.json").is_file():\n                    seen.add(str(hp))\n                    matched_paths.append((hp, "peft_lora_dir"))\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in allowed_exts:\n                matched_paths.append((target, "lora" if target.suffix.lower() in LORA_EXTS else "image"))\n        elif target.is_dir():\n            if include_peft_dirs and (target / "adapter_config.json").is_file():\n                matched_paths.append((target, "peft_lora_dir"))\n            elif not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if not any(resolved.is_relative_to(r) for r in roots):\n                        continue\n                    if resolved.is_file() and resolved.suffix.lower() in allowed_exts:\n                        matched_paths.append((resolved, "lora" if resolved.suffix.lower() in LORA_EXTS else "image"))\n                    elif include_peft_dirs and resolved.is_dir() and (resolved / "adapter_config.json").is_file():\n                        matched_paths.append((resolved, "peft_lora_dir"))\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    cur_root = Path(root_dir).resolve()\n                    if include_peft_dirs and cur_root != target and (cur_root / "adapter_config.json").is_file():\n                        if any(cur_root.is_relative_to(r) for r in roots):\n                            matched_paths.append((cur_root, "peft_lora_dir"))\n                        dirnames[:] = []\n                        continue\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in allowed_exts:\n                            matched_paths.append((hp, "lora" if hp.suffix.lower() in LORA_EXTS else "image"))\n\n    items: list[dict[str, Any]] = []\n    for p, ftype in matched_paths:\n        st = p.stat()\n        size_bytes = st.st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "type": ftype,\n            "size_bytes": size_bytes,\n            "size_mb": round(size_bytes / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef save_output_images_to_drive(\n    images_bytes: list[bytes],\n    spec: str,\n    *,\n    data_dir: Path = DATA_DIR,\n    output_dir: Path = DRIVE_OUTPUTS_DIR,\n    prefix: str = "qwen-image",\n    fmt: str = "png",\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> list[str]:\n    """Persist one or more generated image byte buffers safely inside Google Drive."""\n    if not images_bytes:\n        return []\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "png").lower())\n    if ext not in IMAGE_EXTS:\n        ext = ".png"\n\n    stamp = f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}-{secrets.token_hex(3)}"\n    raw_spec = (spec or "").strip()\n    is_auto = not raw_spec or raw_spec in ("auto", "true", "True", "1")\n    treat_as_dir = False\n\n    if is_auto:\n        base_dir = output_dir\n        stem = stamp\n        treat_as_dir = True\n    else:\n        dest = resolve_drive_path(\n            raw_spec,\n            data_dir=data_dir,\n            default_subdir="outputs",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or raw_spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            base_dir = dest\n            stem = stamp\n            treat_as_dir = True\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n            base_dir = dest.parent\n            stem = dest.stem\n            ext = dest.suffix\n\n    saved: list[str] = []\n    total = len(images_bytes)\n    for idx, img_bytes in enumerate(images_bytes, 1):\n        if total == 1 and not treat_as_dir:\n            out_path = base_dir / f"{stem}{ext}"\n        elif total == 1:\n            out_path = base_dir / f"{stem}{ext}"\n            if out_path.exists():\n                out_path = base_dir / f"{stem}-{int(time.time() * 1000) % 10000:04d}{ext}"\n        else:\n            out_path = base_dir / f"{stem}-{idx:02d}{ext}"\n        out_path.write_bytes(img_bytes)\n        saved.append(str(out_path))\n    return saved\n\n\ndef resolve_and_stage_lora(\n    lora_spec: Any,\n    *,\n    data_dir: Path = DATA_DIR,\n    local_lora_dir: Path = LOCAL_LORA_DIR,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> dict[str, Any]:\n    """Validate a LoRA spec against allowed roots and stage Drive LoRAs onto local SSD."""\n    if isinstance(lora_spec, str):\n        raw_path = lora_spec.strip()\n        raw_name = ""\n        scale = 1.0\n    elif isinstance(lora_spec, dict):\n        raw_path = str(lora_spec.get("path") or "").strip()\n        raw_name = str(lora_spec.get("name") or "").strip()\n        scale = float(lora_spec.get("scale", 1.0))\n    else:\n        raise ValueError("Invalid \'lora\' parameter: expected object {\'path\', \'name\'?, \'scale\'?} or Drive path string")\n\n    if not raw_path:\n        raise ValueError("LoRA \'path\' must not be empty")\n    if "\\x00" in raw_path:\n        raise ValueError("Invalid NUL byte in LoRA path")\n\n    local_root = local_lora_dir.resolve()\n    cand_local = Path(raw_path)\n    if cand_local.is_absolute() and not raw_path.startswith(("/content/drive/", "/drive/")):\n        resolved_local = cand_local.resolve()\n        if resolved_local.is_relative_to(local_root):\n            if not resolved_local.exists():\n                raise FileNotFoundError(f"Staged LoRA not found: {resolved_local}")\n            name = raw_name or resolved_local.stem or resolved_local.name\n            return {"name": name, "path": str(resolved_local), "scale": scale}\n        src_drive = resolve_drive_path(\n            raw_path,\n            data_dir=data_dir,\n            default_subdir="loras",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n    else:\n        bare = raw_path\n        for pfx in ("drive://", "gdrive://", "file://"):\n            if bare.startswith(pfx):\n                bare = ""\n                break\n        if bare and "/" not in bare and ".." not in bare and local_lora_dir.is_dir() and (local_lora_dir / bare).exists():\n            resolved_local = (local_lora_dir / bare).resolve()\n            if resolved_local.is_relative_to(local_root):\n                name = raw_name or resolved_local.stem or resolved_local.name\n                return {"name": name, "path": str(resolved_local), "scale": scale}\n        src_drive = resolve_drive_path(\n            raw_path,\n            data_dir=data_dir,\n            default_subdir="loras",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    if not src_drive.exists():\n        raise FileNotFoundError(f"LoRA adapter not found on Google Drive: {raw_path} (resolved to {src_drive})")\n\n    local_lora_dir.mkdir(parents=True, exist_ok=True)\n    key_hash = hashlib.sha256(str(src_drive).encode("utf-8")).hexdigest()[:10]\n    safe_base = re.sub(r"[^A-Za-z0-9._-]+", "_", src_drive.name)\n    staged_target = local_lora_dir / f"{key_hash}_{safe_base}"\n    stamp_file = local_lora_dir / f".{key_hash}_{safe_base}.stamp"\n\n    st = src_drive.stat()\n    sig = f"{st.st_size}:{st.st_mtime_ns}"\n    if not (staged_target.exists() and stamp_file.is_file() and stamp_file.read_text(encoding="utf-8") == sig):\n        if src_drive.is_file():\n            shutil.copyfile(src_drive, staged_target)\n        elif src_drive.is_dir():\n            if staged_target.exists():\n                shutil.rmtree(staged_target)\n            shutil.copytree(src_drive, staged_target, symlinks=False)\n        else:\n            raise ValueError(f"Unsupported LoRA path type: {src_drive}")\n        stamp_file.write_text(sig, encoding="utf-8")\n\n    name = raw_name or src_drive.stem or src_drive.name\n    return {"name": name, "path": str(staged_target.resolve()), "scale": scale}\n\n\ndef _resolve_image_url_item(\n    raw_url: str,\n    *,\n    data_dir: Path = DATA_DIR,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> str:\n    url = (raw_url or "").strip()\n    if not url:\n        raise ValueError("Empty image URL in messages")\n    if url.startswith(("data:image/", "http://", "https://")):\n        return url\n    img_path = resolve_drive_path(\n        url,\n        data_dir=data_dir,\n        default_subdir="inputs",\n        mydrive_root=mydrive_root,\n        drive_root=drive_root,\n    )\n    return image_path_to_data_url(img_path)\n\n\ndef _extract_drive_options(payload: dict[str, Any], header_save: str = "", header_drive_only: str = "") -> tuple[str, bool]:\n    save_val = payload.pop("save_to_drive", None)\n    if save_val is None:\n        save_val = payload.pop("save_drive_path", None)\n    drive_only_val = payload.pop("drive_only", False)\n\n    save_spec = ""\n    if isinstance(save_val, bool):\n        save_spec = "auto" if save_val else ""\n    elif isinstance(save_val, str) and save_val.strip():\n        save_spec = save_val.strip()\n    if not save_spec and header_save:\n        save_spec = header_save.strip()\n\n    drive_only = bool(drive_only_val) or (header_drive_only.strip().lower() in ("1", "true", "yes"))\n    if drive_only and not save_spec:\n        save_spec = "auto"\n    return save_spec, drive_only\n\n\ndef _normalize_served_model(payload: dict[str, Any]) -> None:\n    """Map short model aliases and common OpenAI image aliases to primary MODEL_REPO for vLLM-Omni."""\n    req_model = str(payload.get("model") or "").strip()\n    accepted_aliases = {\n        "",\n        MODEL_REPO,\n        MODEL_ALIAS,\n        "qwen-image",\n        "qwen-image-2.1",\n        "Qwen-Image-2.1",\n        MODEL_REPO.split("/")[-1],\n        # Seamless drop-in compatibility for OpenAI Images SDK / Studio apps (e.g. open-pose-costume)\n        "gpt-image-1",\n        "gpt-image-1.5",\n        "gpt-image-2",\n    }\n    if req_model in accepted_aliases:\n        payload["model"] = MODEL_REPO\n\n\ndef _apply_qwen_sampling_defaults(payload: dict[str, Any]) -> str | None:\n    """Inject Qwen-Image-2.1\'s recommended 40-step / True CFG defaults and pop custom color_format."""\n    color_format_raw = payload.pop("color_format", None)\n    color_format = str(color_format_raw).strip().upper() if color_format_raw else None\n    if color_format and color_format not in ("RGB", "RGBA"):\n        raise ValueError(f"Invalid color_format \'{color_format_raw}\': expected \'RGB\' or \'RGBA\'")\n\n    if payload.get("steps") is not None and payload.get("num_inference_steps") is None:\n        payload["num_inference_steps"] = int(payload.pop("steps"))\n    else:\n        payload.pop("steps", None)\n    if payload.get("num_inference_steps") is None:\n        payload["num_inference_steps"] = DEFAULT_STEPS\n\n    # Qwen-Image-2.1 uses true_cfg_scale, not guidance_scale.\n    cfg_alias = payload.pop("cfg_scale", None)\n    guidance_val = payload.pop("guidance_scale", None)\n    if payload.get("true_cfg_scale") is None and cfg_alias is not None:\n        payload["true_cfg_scale"] = float(cfg_alias)\n\n    neg = payload.get("negative_prompt")\n    if isinstance(neg, str) and not neg.strip():\n        payload.pop("negative_prompt", None)\n        neg = None\n\n    if payload.get("true_cfg_scale") is None:\n        if neg:\n            payload["true_cfg_scale"] = (\n                float(guidance_val) if guidance_val is not None else DEFAULT_TRUE_CFG_WITH_NEG\n            )\n        else:\n            payload["true_cfg_scale"] = DEFAULT_TRUE_CFG_NO_NEG\n\n    # Strip OpenAI SDK \'quality\' if sent by generic clients so vLLM-Omni doesn\'t reject/warn\n    payload.pop("quality", None)\n    return color_format\n\n\ndef _prepare_generation_request(\n    raw_body: bytes,\n    *,\n    header_save: str = "",\n    header_drive_only: str = "",\n    data_dir: Path = DATA_DIR,\n    local_lora_dir: Path = LOCAL_LORA_DIR,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> tuple[bytes, str, bool, str | None]:\n    """Validate and rewrite a POST /v1/images/generations JSON body."""\n    try:\n        payload = json.loads(raw_body)\n    except Exception as err:\n        raise ValueError(f"Invalid JSON request body: {err}") from err\n    if not isinstance(payload, dict):\n        raise ValueError("JSON request body must be an object")\n\n    _normalize_served_model(payload)\n    save_spec, drive_only = _extract_drive_options(payload, header_save, header_drive_only)\n    color_format = _apply_qwen_sampling_defaults(payload)\n\n    if "size" in payload and payload["size"] is not None:\n        w, h = parse_size_spec(str(payload["size"]), allow_auto=True)\n        if w is None or h is None:\n            payload["size"] = "1024x1024"\n        else:\n            payload["size"] = f"{w}x{h}"\n    elif payload.get("width") is not None or payload.get("height") is not None:\n        w_in = int(payload.pop("width", 1024))\n        h_in = int(payload.pop("height", 1024))\n        w, h = parse_size_spec(f"{w_in}x{h_in}")\n        payload["size"] = f"{w}x{h}"\n    else:\n        payload["size"] = "1024x1024"\n\n    if payload.get("lora") is not None:\n        payload["lora"] = resolve_and_stage_lora(\n            payload["lora"],\n            data_dir=data_dir,\n            local_lora_dir=local_lora_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    if save_spec and save_spec not in ("auto", "true", "True", "1"):\n        resolve_drive_path(\n            save_spec,\n            data_dir=data_dir,\n            default_subdir="outputs",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    if save_spec and payload.get("response_format") not in (None, "b64_json"):\n        payload["response_format"] = "b64_json"\n\n    return json.dumps(payload).encode("utf-8"), save_spec, drive_only, color_format\n\n\ndef _rewrite_drive_urls_in_chat_body(\n    raw_body: bytes,\n    *,\n    header_save: str = "",\n    header_drive_only: str = "",\n    data_dir: Path = DATA_DIR,\n    local_lora_dir: Path = LOCAL_LORA_DIR,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> tuple[bytes, str, bool, str | None]:\n    """Validate and rewrite Drive image references and LoRAs in POST /v1/chat/completions."""\n    try:\n        payload = json.loads(raw_body)\n    except Exception as err:\n        raise ValueError(f"Invalid JSON request body: {err}") from err\n    if not isinstance(payload, dict):\n        raise ValueError("JSON request body must be an object")\n\n    extra_body = payload.get("extra_body")\n    if isinstance(extra_body, dict):\n        for k, v in list(extra_body.items()):\n            payload.setdefault(k, v)\n        payload.pop("extra_body", None)\n\n    _normalize_served_model(payload)\n    save_spec, drive_only = _extract_drive_options(payload, header_save, header_drive_only)\n    color_format = _apply_qwen_sampling_defaults(payload)\n\n    if "size" in payload and payload["size"] is not None:\n        w, h = parse_size_spec(str(payload.pop("size")), allow_auto=True)\n        if w is not None and h is not None:\n            payload["width"] = w\n            payload["height"] = h\n    elif payload.get("width") is not None or payload.get("height") is not None:\n        w_in = int(payload.get("width") or 1024)\n        h_in = int(payload.get("height") or 1024)\n        w, h = parse_size_spec(f"{w_in}x{h_in}")\n        payload["width"] = w\n        payload["height"] = h\n\n    if payload.get("lora") is not None:\n        payload["lora"] = resolve_and_stage_lora(\n            payload["lora"],\n            data_dir=data_dir,\n            local_lora_dir=local_lora_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    if save_spec and save_spec not in ("auto", "true", "True", "1"):\n        resolve_drive_path(\n            save_spec,\n            data_dir=data_dir,\n            default_subdir="outputs",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    resolved_images: list[str] = []\n    for msg in payload.get("messages") or []:\n        if not isinstance(msg, dict):\n            continue\n        content = msg.get("content")\n        if not isinstance(content, list):\n            continue\n        for item in content:\n            if not isinstance(item, dict) or item.get("type") != "image_url":\n                continue\n            iurl = item.get("image_url")\n            if isinstance(iurl, str):\n                item["image_url"] = {"url": iurl}\n                iurl = item["image_url"]\n            if not isinstance(iurl, dict) or not isinstance(iurl.get("url"), str):\n                raise ValueError("Invalid image_url entry in messages")\n            resolved_url = _resolve_image_url_item(\n                iurl["url"],\n                data_dir=data_dir,\n                mydrive_root=mydrive_root,\n                drive_root=drive_root,\n            )\n            iurl["url"] = resolved_url\n            resolved_images.append(resolved_url)\n\n    if len(resolved_images) > MAX_REFERENCE_IMAGES:\n        raise ValueError(\n            f"Qwen-Image-2.1 supports at most {MAX_REFERENCE_IMAGES} reference images (<image1>..<image4>), "\n            f"got {len(resolved_images)}"\n        )\n\n    return json.dumps(payload).encode("utf-8"), save_spec, drive_only, color_format\n\n\ndef _extract_images_from_generations_response(resp_obj: dict[str, Any]) -> list[tuple[dict[str, Any], bytes, str]]:\n    out: list[tuple[dict[str, Any], bytes, str]] = []\n    for item in resp_obj.get("data") or []:\n        if not isinstance(item, dict):\n            continue\n        b64 = item.get("b64_json")\n        if isinstance(b64, str) and b64:\n            raw = base64.b64decode(b64)\n            mime = detect_image_mime(raw)\n            out.append((item, raw, EXT_BY_MIME.get(mime, "png")))\n    return out\n\n\ndef _extract_images_from_chat_response(resp_obj: dict[str, Any]) -> list[tuple[dict[str, Any], bytes, str]]:\n    out: list[tuple[dict[str, Any], bytes, str]] = []\n    for choice in resp_obj.get("choices") or []:\n        msg = choice.get("message") if isinstance(choice, dict) else None\n        content = msg.get("content") if isinstance(msg, dict) else None\n        if not isinstance(content, list):\n            continue\n        for part in content:\n            if not isinstance(part, dict) or part.get("type") != "image_url":\n                continue\n            iurl = part.get("image_url")\n            url_val = iurl.get("url") if isinstance(iurl, dict) else None\n            if isinstance(url_val, str) and url_val.startswith("data:image/") and "," in url_val:\n                header, b64 = url_val.split(",", 1)\n                raw = base64.b64decode(b64)\n                mime = header.split(";")[0].removeprefix("data:") or detect_image_mime(raw)\n                out.append((iurl, raw, EXT_BY_MIME.get(mime, "png")))\n    return out\n\n\ndef _parse_multipart_edit_body(content_type: str, raw_body: bytes) -> dict[str, Any]:\n    """Parse multipart/form-data (e.g. OpenAI SDK / curl -F / browser FormData) preserving exact image order."""\n    raw_msg = (\n        f"Content-Type: {content_type}\\r\\nMIME-Version: 1.0\\r\\n\\r\\n".encode("utf-8")\n        + raw_body\n    )\n    msg = BytesParser(policy=email_default_policy).parsebytes(raw_msg)\n    if not msg.is_multipart():\n        raise ValueError("Invalid multipart/form-data payload")\n\n    fields: dict[str, Any] = {}\n    ordered_inputs: list[tuple[str, Any]] = []\n    for part in msg.iter_parts():\n        disp = part.get("Content-Disposition", "")\n        if "form-data" not in disp:\n            continue\n        name = part.get_param("name", header="content-disposition")\n        filename = part.get_filename()\n        payload_bytes = part.get_payload(decode=True) or b""\n        if not name:\n            continue\n        clean_name = name.rstrip("[]")\n        if filename is not None or (clean_name in ("image", "images") and part.get_content_type().startswith("image/")):\n            if not payload_bytes:\n                continue\n            mime = detect_image_mime(payload_bytes, filename or "upload.png")\n            ordered_inputs.append(("bytes", (payload_bytes, filename or f"image.{EXT_BY_MIME.get(mime, \'png\')}", mime)))\n        else:\n            text_val = payload_bytes.decode("utf-8", errors="replace").strip()\n            if clean_name in ("image", "images", "url", "drive_image", "drive_images"):\n                if text_val:\n                    ordered_inputs.append(("spec", text_val))\n            else:\n                fields[clean_name] = text_val\n\n    if ordered_inputs:\n        fields["_ordered_inputs"] = ordered_inputs\n    for int_key in ("seed", "num_inference_steps", "steps", "width", "height", "n", "output_compression"):\n        if int_key in fields and fields[int_key] != "":\n            fields[int_key] = int(fields[int_key])\n    for float_key in ("true_cfg_scale", "cfg_scale", "guidance_scale", "lora_scale"):\n        if float_key in fields and fields[float_key] != "":\n            fields[float_key] = float(fields[float_key])\n    for bool_key in ("drive_only",):\n        if bool_key in fields:\n            fields[bool_key] = str(fields[bool_key]).lower() in ("1", "true", "yes")\n    if "lora" in fields and isinstance(fields["lora"], str) and fields["lora"].startswith("{"):\n        fields["lora"] = json.loads(fields["lora"])\n    return fields\n\n\ndef _encode_multipart_formdata(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]],\n) -> tuple[str, bytes]:\n    """Build a multipart/form-data body for loopback forwarding to vLLM-Omni /v1/images/edits."""\n    boundary = f"----ColabQwenImageBoundary{secrets.token_hex(12)}"\n    buf = bytearray()\n    for fname, fval in fields:\n        buf.extend(f"--{boundary}\\r\\n".encode("ascii"))\n        buf.extend(f\'Content-Disposition: form-data; name="{fname}"\\r\\n\\r\\n\'.encode("utf-8"))\n        buf.extend(str(fval).encode("utf-8"))\n        buf.extend(b"\\r\\n")\n    for field_name, filename, mime, raw in files:\n        safe_fn = re.sub(r\'["\\r\\n\\\\]\', "_", filename or "image.png")\n        buf.extend(f"--{boundary}\\r\\n".encode("ascii"))\n        buf.extend(\n            f\'Content-Disposition: form-data; name="{field_name}"; filename="{safe_fn}"\\r\\n\'.encode("utf-8")\n        )\n        buf.extend(f"Content-Type: {mime}\\r\\n\\r\\n".encode("ascii"))\n        buf.extend(raw)\n        buf.extend(b"\\r\\n")\n    buf.extend(f"--{boundary}--\\r\\n".encode("ascii"))\n    return f"multipart/form-data; boundary={boundary}", bytes(buf)\n\n\ndef _resolve_edit_input_item(\n    kind: str,\n    item: Any,\n    idx: int,\n    *,\n    data_dir: Path,\n    mydrive_root: Path,\n    drive_root: Path,\n) -> tuple[str, Any]:\n    """Resolve an edit input into either (\'file\', (filename, mime, bytes)) or (\'http\', url_str)."""\n    if kind == "bytes":\n        raw_bytes, filename, mime = item\n        return "file", (filename, mime, raw_bytes)\n    spec = str(item or "").strip()\n    if not spec:\n        raise ValueError("Empty reference image entry")\n    if spec.startswith(("http://", "https://")):\n        return "http", spec\n    if spec.startswith("data:image/") and "," in spec:\n        header, b64 = spec.split(",", 1)\n        raw_bytes = base64.b64decode(b64)\n        if not raw_bytes:\n            raise ValueError("Decoded data:image URI is empty")\n        mime = header.split(";")[0].removeprefix("data:") or detect_image_mime(raw_bytes)\n        ext = EXT_BY_MIME.get(mime, "png")\n        return "file", (f"ref_{idx}.{ext}", mime, raw_bytes)\n    img_path = resolve_drive_path(\n        spec,\n        data_dir=data_dir,\n        default_subdir="inputs",\n        mydrive_root=mydrive_root,\n        drive_root=drive_root,\n    )\n    if not img_path.is_file():\n        raise FileNotFoundError(f"Image file not found on Google Drive: {img_path}")\n    raw_bytes = img_path.read_bytes()\n    if not raw_bytes:\n        raise ValueError(f"Image file is empty: {img_path}")\n    mime = detect_image_mime(raw_bytes, img_path.name)\n    return "file", (img_path.name, mime, raw_bytes)\n\n\ndef _prepare_edit_request(\n    content_type: str,\n    raw_body: bytes,\n    *,\n    header_save: str = "",\n    header_drive_only: str = "",\n    data_dir: Path = DATA_DIR,\n    local_lora_dir: Path = LOCAL_LORA_DIR,\n    mydrive_root: Path = MYDRIVE_ROOT,\n    drive_root: Path = DRIVE_ROOT,\n) -> tuple[str, str, bytes, str, bool, str | None]:\n    """Prepare POST /v1/images/edits or POST /api/ct/edit for direct in-process ASGI dispatch.\n\n    Returns (target_path, new_content_type, new_body_bytes, save_spec, drive_only, color_format).\n    Pure binary/Drive/data-URI or pure HTTP-URL batches rewrite into multipart/form-data for upstream\n    /v1/images/edits (preserving 4-channel RGBA input PNGs, size=\'auto\', and multi-MB uploads with\n    zero loopback HTTP calls). Mixed HTTP-URL + binary/Drive batches rewrite into /v1/chat/completions\n    JSON to preserve exact <image1>..<image4> ordering without middleware SSRF fetches.\n    """\n    if "multipart/form-data" in (content_type or "").lower():\n        body = _parse_multipart_edit_body(content_type, raw_body)\n    else:\n        try:\n            body = json.loads(raw_body)\n        except Exception as err:\n            raise ValueError(f"Invalid JSON body: {err}") from err\n        if not isinstance(body, dict):\n            raise ValueError("Request body must be a JSON object")\n\n    prompt = str(body.get("prompt") or "").strip()\n    if not prompt:\n        raise ValueError("Missing required \'prompt\'")\n\n    ordered_raw: list[tuple[str, Any]] = list(body.pop("_ordered_inputs", None) or [])\n    if not ordered_raw:\n        for key in ("images", "image", "drive_images", "drive_image", "url"):\n            val = body.get(key)\n            if isinstance(val, str) and val.strip():\n                ordered_raw.append(("spec", val.strip()))\n            elif isinstance(val, list):\n                for x in val:\n                    if str(x).strip():\n                        ordered_raw.append(("spec", str(x).strip()))\n\n    if not ordered_raw:\n        raise ValueError("At least 1 reference image is required (\'image\' or \'images\')")\n    if len(ordered_raw) > MAX_REFERENCE_IMAGES:\n        raise ValueError(\n            f"Qwen-Image-2.1 supports at most {MAX_REFERENCE_IMAGES} reference images (<image1>..<image4>), "\n            f"got {len(ordered_raw)}"\n        )\n\n    _normalize_served_model(body)\n    save_spec, drive_only = _extract_drive_options(body, header_save, header_drive_only)\n    color_format = _apply_qwen_sampling_defaults(body)\n\n    if save_spec and save_spec not in ("auto", "true", "True", "1"):\n        resolve_drive_path(\n            save_spec,\n            data_dir=data_dir,\n            default_subdir="outputs",\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    staged_lora: dict[str, Any] | None = None\n    if body.get("lora") is not None:\n        staged_lora = resolve_and_stage_lora(\n            body["lora"],\n            data_dir=data_dir,\n            local_lora_dir=local_lora_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n    elif body.get("lora_path"):\n        staged_lora = resolve_and_stage_lora(\n            {\n                "path": body["lora_path"],\n                "name": body.get("lora_name") or "",\n                "scale": float(body.get("lora_scale", 1.0)),\n            },\n            data_dir=data_dir,\n            local_lora_dir=local_lora_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n\n    size_str = "auto"\n    if body.get("size") is not None:\n        w, h = parse_size_spec(str(body["size"]), allow_auto=True)\n        size_str = "auto" if (w is None or h is None) else f"{w}x{h}"\n    elif body.get("width") is not None or body.get("height") is not None:\n        w_in = int(body.get("width") or 1024)\n        h_in = int(body.get("height") or 1024)\n        w, h = parse_size_spec(f"{w_in}x{h_in}")\n        size_str = f"{w}x{h}"\n\n    resolved_items = [\n        _resolve_edit_input_item(\n            kind,\n            item,\n            idx,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        for idx, (kind, item) in enumerate(ordered_raw, 1)\n    ]\n    has_files = any(rk == "file" for rk, _ in resolved_items)\n    has_http = any(rk == "http" for rk, _ in resolved_items)\n\n    if has_files and has_http:\n        content_parts: list[dict[str, Any]] = []\n        for rk, rv in resolved_items:\n            if rk == "http":\n                content_parts.append({"type": "image_url", "image_url": {"url": rv}})\n            else:\n                _, mime, raw_b = rv\n                b64_u = f"data:{mime};base64,{base64.b64encode(raw_b).decode(\'ascii\')}"\n                content_parts.append({"type": "image_url", "image_url": {"url": b64_u}})\n        content_parts.append({"type": "text", "text": prompt})\n        chat_payload: dict[str, Any] = {\n            "model": MODEL_REPO,\n            "messages": [{"role": "user", "content": content_parts}],\n            "num_inference_steps": int(body["num_inference_steps"]),\n            "true_cfg_scale": float(body["true_cfg_scale"]),\n        }\n        if size_str != "auto":\n            w_s, h_s = size_str.split("x", 1)\n            chat_payload["width"] = int(w_s)\n            chat_payload["height"] = int(h_s)\n        if body.get("negative_prompt"):\n            chat_payload["negative_prompt"] = str(body["negative_prompt"])\n        if body.get("seed") is not None:\n            chat_payload["seed"] = int(body["seed"])\n        if staged_lora is not None:\n            chat_payload["lora"] = staged_lora\n        return (\n            "/v1/chat/completions",\n            "application/json",\n            json.dumps(chat_payload).encode("utf-8"),\n            save_spec,\n            drive_only,\n            color_format,\n        )\n\n    form_fields: list[tuple[str, str]] = [\n        ("model", MODEL_REPO),\n        ("prompt", prompt),\n        ("size", size_str),\n        ("response_format", "b64_json"),\n        ("num_inference_steps", str(int(body["num_inference_steps"]))),\n        ("true_cfg_scale", str(float(body["true_cfg_scale"]))),\n    ]\n    if body.get("n") is not None:\n        form_fields.append(("n", str(int(body["n"]))))\n    if body.get("seed") is not None:\n        form_fields.append(("seed", str(int(body["seed"]))))\n    if body.get("negative_prompt"):\n        form_fields.append(("negative_prompt", str(body["negative_prompt"])))\n    if body.get("output_format"):\n        form_fields.append(("output_format", str(body["output_format"])))\n    if body.get("background"):\n        form_fields.append(("background", str(body["background"])))\n    if body.get("output_compression") is not None:\n        form_fields.append(("output_compression", str(int(body["output_compression"]))))\n    if staged_lora is not None:\n        form_fields.append(("lora", json.dumps(staged_lora)))\n\n    form_files: list[tuple[str, str, str, bytes]] = []\n    for rk, rv in resolved_items:\n        if rk == "http":\n            form_fields.append(("url", str(rv)))\n        else:\n            fn, mime, raw_b = rv\n            form_files.append(("image", fn, mime, raw_b))\n\n    mp_ct, mp_bytes = _encode_multipart_formdata(form_fields, form_files)\n    return "/v1/images/edits", mp_ct, mp_bytes, save_spec, drive_only, color_format\n\n\nasync def _read_full_body(receive: Receive) -> bytes | None:\n    chunks: list[bytes] = []\n    while True:\n        msg = await receive()\n        if msg["type"] == "http.request":\n            chunks.append(msg.get("body", b""))\n            if not msg.get("more_body", False):\n                return b"".join(chunks)\n        elif msg["type"] == "http.disconnect":\n            return None\n\n\ndef _with_rewritten_request(scope: Scope, *, path: str, content_type: str, length: int) -> Scope:\n    new_scope = dict(scope)\n    new_scope["path"] = path\n    new_scope["raw_path"] = path.encode("utf-8")\n    raw_headers = [\n        (k, v)\n        for (k, v) in scope.get("headers", [])\n        if k.lower() not in (b"content-length", b"content-type")\n    ]\n    raw_headers.append((b"content-type", content_type.encode("utf-8")))\n    raw_headers.append((b"content-length", str(length).encode("ascii")))\n    new_scope["headers"] = raw_headers\n    return new_scope\n\n\ndef _inject_cors_headers(raw_headers: list[tuple[bytes, bytes]]) -> list[tuple[bytes, bytes]]:\n    existing = {k.lower() for k, _ in raw_headers}\n    merged = list(raw_headers)\n    for ck, cv in CORS_HEADERS:\n        if ck not in existing:\n            merged.append((ck, cv))\n    return merged\n\n\nclass QwenImageMiddleware:\n    """ASGI middleware enforcing Bearer auth, CORS preflight, Google Drive I/O, and Qwen-Image-2.1 defaults."""\n\n    def __init__(self, app: ASGIApp) -> None:\n        self.app = app\n\n    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n        if scope["type"] not in ("http", "websocket"):\n            await self.app(scope, receive, send)\n            return\n\n        if scope["type"] == "http" and scope.get("method") == "OPTIONS":\n            resp = Response(status_code=HTTPStatus.NO_CONTENT)\n            async def cors_options_send(message: Message) -> None:\n                if message["type"] == "http.response.start":\n                    message = dict(message)\n                    message["headers"] = _inject_cors_headers(list(message.get("headers", [])))\n                await send(message)\n            await resp(scope, receive, cors_options_send)\n            return\n\n        async def send_with_cors(message: Message) -> None:\n            if message["type"] == "http.response.start":\n                message = dict(message)\n                message["headers"] = _inject_cors_headers(list(message.get("headers", [])))\n            await send(message)\n\n        root_path = scope.get("root_path", "")\n        path = scope["path"].removeprefix(root_path)\n        headers = Headers(scope=scope)\n\n        if not _verify_auth(headers):\n            resp = JSONResponse(\n                {\n                    "error": {\n                        "message": "Unauthorized: invalid or missing Bearer token",\n                        "type": "AuthenticationError",\n                        "code": 401,\n                    }\n                },\n                status_code=HTTPStatus.UNAUTHORIZED,\n            )\n            await resp(scope, receive, send_with_cors)\n            return\n\n        if scope["type"] != "http":\n            await self.app(scope, receive, send)\n            return\n\n        method = scope.get("method", "GET")\n\n        if method == "GET" and path == "/api/ct/info":\n            served = list(dict.fromkeys([MODEL_REPO, MODEL_ALIAS, "qwen-image", "qwen-image-2.1"]))\n            resp = JSONResponse({\n                "service": "qwen-image",\n                "model_repo": MODEL_REPO,\n                "served_models": served,\n                "quantization": QUANTIZATION,\n                "prefix_kv_cache_dtype": PREFIX_KV_DTYPE,\n                "step_execution": STEP_EXECUTION,\n                "vae_use_tiling": VAE_TILING,\n                "max_num_seqs": MAX_NUM_SEQS,\n                "default_steps": DEFAULT_STEPS,\n                "default_true_cfg_scale": DEFAULT_TRUE_CFG_NO_NEG,\n                "default_true_cfg_scale_with_negative_prompt": DEFAULT_TRUE_CFG_WITH_NEG,\n                "default_size": "1024x1024",\n                "dimension_alignment": DIM_ALIGNMENT,\n                "rgba_vae": True,\n                "max_reference_images": MAX_REFERENCE_IMAGES,\n                "preloaded_lora": PRELOADED_LORA or None,\n                "drive_mounted": MYDRIVE_ROOT.is_dir(),\n                "data_dir": str(DATA_DIR),\n                "drive_inputs_dir": str(DRIVE_INPUTS_DIR),\n                "drive_outputs_dir": str(DRIVE_OUTPUTS_DIR),\n                "drive_loras_dir": str(DRIVE_LORAS_DIR),\n            })\n            await resp(scope, receive, send_with_cors)\n            return\n\n        if method == "GET" and path == "/api/ct/logs":\n            qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n            try:\n                n_lines = int((qs.get("lines") or ["120"])[0])\n            except ValueError:\n                n_lines = 120\n            resp = JSONResponse(tail_service_log(log_file=Path("/content/ct/service.log"), lines=n_lines))\n            await resp(scope, receive, send_with_cors)\n            return\n\n        if method == "GET" and path == "/api/ct/drive-list":\n            qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n            spec = (qs.get("path") or [""])[0]\n            kind = (qs.get("kind") or ["images"])[0]\n            recursive = (qs.get("recursive") or ["1"])[0] not in ("0", "false", "False")\n            try:\n                files = await asyncio.to_thread(\n                    list_drive_files_by_kind,\n                    spec,\n                    kind=kind,\n                    data_dir=DATA_DIR,\n                    recursive=recursive,\n                    mydrive_root=MYDRIVE_ROOT,\n                    drive_root=DRIVE_ROOT,\n                )\n                resp = JSONResponse({\n                    "drive_mounted": MYDRIVE_ROOT.is_dir(),\n                    "kind": kind,\n                    "query": spec,\n                    "count": len(files),\n                    "files": files,\n                })\n            except ValueError as err:\n                resp = JSONResponse({"error": str(err)}, status_code=400)\n            except Exception as err:\n                resp = JSONResponse({"error": str(err)}, status_code=500)\n            await resp(scope, receive, send_with_cors)\n            return\n\n        if method == "POST" and path in (\n            "/api/ct/edit",\n            "/v1/images/edits",\n            "/v1/images/generations",\n            "/v1/chat/completions",\n        ):\n            raw_body = await _read_full_body(receive)\n            if raw_body is None:\n                return\n            ct = headers.get("content-type", "application/json")\n            h_save = headers.get("x-save-drive", "")\n            h_only = headers.get("x-drive-only", "")\n            is_edit_route = path in ("/api/ct/edit", "/v1/images/edits")\n\n            try:\n                if is_edit_route:\n                    target_path, new_ct, rewritten_body, save_spec, drive_only, color_format = await asyncio.to_thread(\n                        _prepare_edit_request,\n                        ct,\n                        raw_body,\n                        header_save=h_save,\n                        header_drive_only=h_only,\n                        data_dir=DATA_DIR,\n                        local_lora_dir=LOCAL_LORA_DIR,\n                        mydrive_root=MYDRIVE_ROOT,\n                        drive_root=DRIVE_ROOT,\n                    )\n                    wants_images_shape = True\n                    save_prefix = "qwen-edit"\n                elif path == "/v1/images/generations":\n                    rewritten_body, save_spec, drive_only, color_format = await asyncio.to_thread(\n                        _prepare_generation_request,\n                        raw_body,\n                        header_save=h_save,\n                        header_drive_only=h_only,\n                        data_dir=DATA_DIR,\n                        local_lora_dir=LOCAL_LORA_DIR,\n                        mydrive_root=MYDRIVE_ROOT,\n                        drive_root=DRIVE_ROOT,\n                    )\n                    target_path = "/v1/images/generations"\n                    new_ct = "application/json"\n                    wants_images_shape = True\n                    save_prefix = "qwen-t2i"\n                else:\n                    rewritten_body, save_spec, drive_only, color_format = await asyncio.to_thread(\n                        _rewrite_drive_urls_in_chat_body,\n                        raw_body,\n                        header_save=h_save,\n                        header_drive_only=h_only,\n                        data_dir=DATA_DIR,\n                        local_lora_dir=LOCAL_LORA_DIR,\n                        mydrive_root=MYDRIVE_ROOT,\n                        drive_root=DRIVE_ROOT,\n                    )\n                    target_path = "/v1/chat/completions"\n                    new_ct = "application/json"\n                    wants_images_shape = False\n                    save_prefix = "qwen-i2i"\n            except (ValueError, FileNotFoundError) as err:\n                status = 404 if isinstance(err, FileNotFoundError) else 400\n                resp = JSONResponse(\n                    {"error": {"message": str(err), "type": "BadRequestError", "code": status}},\n                    status_code=status,\n                )\n                await resp(scope, receive, send_with_cors)\n                return\n\n            new_scope = _with_rewritten_request(\n                scope,\n                path=root_path + target_path,\n                content_type=new_ct,\n                length=len(rewritten_body),\n            )\n            body_sent = False\n\n            async def replay_receive() -> Message:\n                nonlocal body_sent\n                if not body_sent:\n                    body_sent = True\n                    return {"type": "http.request", "body": rewritten_body, "more_body": False}\n                return await receive()\n\n            upstream_is_images = target_path in ("/v1/images/generations", "/v1/images/edits")\n            convert_chat_to_images = wants_images_shape and not upstream_is_images\n            need_postprocess = bool(save_spec or color_format == "RGB" or convert_chat_to_images)\n            if not need_postprocess:\n                await self.app(new_scope, replay_receive, send_with_cors)\n                return\n\n            resp_status = 200\n            resp_headers: list[tuple[bytes, bytes]] = []\n            resp_chunks: list[bytes] = []\n            start_msg: Message | None = None\n\n            async def tee_send(message: Message) -> None:\n                nonlocal resp_status, resp_headers, start_msg\n                if message["type"] == "http.response.start":\n                    resp_status = message.get("status", 200)\n                    resp_headers = list(message.get("headers", []))\n                    start_msg = message\n                    if resp_status != 200:\n                        await send_with_cors(message)\n                elif message["type"] == "http.response.body":\n                    chunk = message.get("body", b"")\n                    more = message.get("more_body", False)\n                    if resp_status != 200:\n                        await send_with_cors(message)\n                        return\n                    if chunk:\n                        resp_chunks.append(chunk)\n                    if not more and start_msg is not None:\n                        full_raw = b"".join(resp_chunks)\n                        saved_paths: list[str] = []\n                        out_bytes = full_raw\n                        try:\n                            resp_obj = json.loads(full_raw.decode("utf-8"))\n                            extracted = (\n                                _extract_images_from_generations_response(resp_obj)\n                                if upstream_is_images\n                                else _extract_images_from_chat_response(resp_obj)\n                            )\n                            if extracted and color_format == "RGB":\n                                updated_extracted: list[tuple[dict[str, Any], bytes, str]] = []\n                                for holder, img_bytes, fmt in extracted:\n                                    rgb_bytes = await asyncio.to_thread(\n                                        _maybe_convert_color_format, img_bytes, "RGB"\n                                    )\n                                    b64_rgb = base64.b64encode(rgb_bytes).decode("ascii")\n                                    if upstream_is_images:\n                                        holder["b64_json"] = b64_rgb\n                                    else:\n                                        holder["url"] = f"data:image/png;base64,{b64_rgb}"\n                                    updated_extracted.append((holder, rgb_bytes, fmt))\n                                extracted = updated_extracted\n\n                            if save_spec and extracted:\n                                saved_paths = await asyncio.to_thread(\n                                    save_output_images_to_drive,\n                                    [raw for _, raw, _ in extracted],\n                                    save_spec,\n                                    data_dir=DATA_DIR,\n                                    output_dir=DRIVE_OUTPUTS_DIR,\n                                    prefix=save_prefix,\n                                    fmt=extracted[0][2],\n                                    mydrive_root=MYDRIVE_ROOT,\n                                    drive_root=DRIVE_ROOT,\n                                )\n\n                            if convert_chat_to_images:\n                                data_entries: list[dict[str, Any]] = []\n                                for idx, (_, img_bytes, _) in enumerate(extracted):\n                                    entry: dict[str, Any] = {}\n                                    if not drive_only:\n                                        entry["b64_json"] = base64.b64encode(img_bytes).decode("ascii")\n                                    if idx < len(saved_paths):\n                                        sp = Path(saved_paths[idx])\n                                        entry["saved_drive_path"] = str(sp)\n                                        entry["rel_drive_path"] = rel_to_mydrive(sp, MYDRIVE_ROOT)\n                                        entry["size_bytes"] = len(img_bytes)\n                                    data_entries.append(entry)\n                                resp_obj = {"created": int(time.time()), "data": data_entries}\n                                if saved_paths:\n                                    resp_obj["saved_drive_paths"] = saved_paths\n                                    resp_obj["rel_drive_paths"] = [\n                                        rel_to_mydrive(Path(p), MYDRIVE_ROOT) for p in saved_paths\n                                    ]\n                            elif save_spec and extracted:\n                                resp_obj["saved_drive_paths"] = saved_paths\n                                resp_obj["rel_drive_paths"] = [\n                                    rel_to_mydrive(Path(p), MYDRIVE_ROOT) for p in saved_paths\n                                ]\n                                for idx, (holder, img_bytes, _) in enumerate(extracted):\n                                    if idx < len(saved_paths):\n                                        holder["saved_drive_path"] = saved_paths[idx]\n                                        holder["rel_drive_path"] = rel_to_mydrive(Path(saved_paths[idx]), MYDRIVE_ROOT)\n                                        holder["size_bytes"] = len(img_bytes)\n                                    if drive_only:\n                                        if upstream_is_images:\n                                            holder.pop("b64_json", None)\n                                        else:\n                                            holder["url"] = f"drive://{rel_to_mydrive(Path(saved_paths[idx]), MYDRIVE_ROOT)}"\n                            out_bytes = json.dumps(resp_obj).encode("utf-8")\n                        except Exception:\n                            out_bytes = full_raw\n\n                        new_headers = [\n                            (k, v)\n                            for (k, v) in resp_headers\n                            if k.lower() not in (b"content-length", b"x-saved-drive-path")\n                        ]\n                        new_headers.append((b"content-length", str(len(out_bytes)).encode("ascii")))\n                        if saved_paths:\n                            safe_hdr = sanitize_header_param(saved_paths[0]).encode("utf-8")\n                            new_headers.append((b"x-saved-drive-path", safe_hdr))\n                        start_msg["headers"] = new_headers\n                        await send_with_cors(start_msg)\n                        await send_with_cors({"type": "http.response.body", "body": out_bytes, "more_body": False})\n                else:\n                    await send_with_cors(message)\n\n            await self.app(new_scope, replay_receive, tee_send)\n            return\n\n        await self.app(scope, receive, send_with_cors)\n', encoding="utf-8")
import ct_qwen_image_middleware as _qi_mw

PRELOADED_LORA_LOCAL = ""
if LORA_PATH_SPEC:
    _staged = _qi_mw.resolve_and_stage_lora(
        {"path": LORA_PATH_SPEC, "scale": float(LORA_SCALE)},
        data_dir=Path(DATA_DIR),
        local_lora_dir=LOCAL_LORA_DIR,
    )
    PRELOADED_LORA_LOCAL = _staged["path"]
    qilog(f"Staged preload LoRA adapter: {PRELOADED_LORA_LOCAL} (scale={LORA_SCALE})")

SERVED_NAMES = list(dict.fromkeys([MODEL_REPO, MODEL_ALIAS, "qwen-image", "qwen-image-2.1"]))

VLLM_ENV = {
    "PATH": f"{VENV_DIR}/bin:/usr/local/cuda/bin:{os.environ.get('PATH', '')}",
    "PYTHONPATH": f"{QI_WORK}:{os.environ.get('PYTHONPATH', '')}",
    "VIRTUAL_ENV": str(VENV_DIR),
    "VLLM_NO_USAGE_STATS": "1",
    "DO_NOT_TRACK": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "FLASHINFER_DISABLE_VERSION_CHECK": "1",
    "FLASHINFER_WORKSPACE_BASE": str(FLASHINFER_CACHE_DIR),
    "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
    "VLLM_CACHE_ROOT": str(VLLM_CACHE_DIR),
    "TORCHINDUCTOR_CACHE_DIR": str(VLLM_CACHE_DIR / "inductor"),
    "CT_API_TOKEN": API_TOKEN,
    "CT_SERVICE_PORT": str(SERVICE_PORT),
    "CT_DATA_DIR": str(DATA_DIR),
    "CT_QWEN_IMAGE_WORK_DIR": str(QI_WORK),
    "CT_QWEN_IMAGE_MODEL_DIR": str(MODEL_DIR),
    "CT_QWEN_IMAGE_LOCAL_LORA_DIR": str(LOCAL_LORA_DIR),
    "CT_QWEN_IMAGE_MODEL": MODEL_REPO,
    "CT_QWEN_IMAGE_ALIAS": MODEL_ALIAS,
    "CT_QWEN_IMAGE_QUANTIZATION": QUANTIZATION,
    "CT_QWEN_IMAGE_PREFIX_KV_DTYPE": PREFIX_KV_DTYPE,
    "CT_QWEN_IMAGE_STEP_EXECUTION": "1" if STEP_EXECUTION else "0",
    "CT_QWEN_IMAGE_VAE_TILING": "1" if VAE_TILING else "0",
    "CT_QWEN_IMAGE_MAX_NUM_SEQS": str(MAX_NUM_SEQS),
    "CT_QWEN_IMAGE_PRELOADED_LORA": PRELOADED_LORA_LOCAL,
}
if _gpu_arch.startswith("12.") and QUANTIZATION != "none":
    VLLM_ENV["VLLM_DISABLED_KERNELS"] = "FlashInferFP8ScaledMMLinearKernel"

SERVER_ARGS = [
    str(VENV_DIR / "bin" / "vllm"), "serve", str(MODEL_DIR),
    "--omni",
    "--served-model-name", *SERVED_NAMES,
    "--host", "127.0.0.1",
    "--port", str(SERVICE_PORT),
    "--api-key", API_TOKEN,
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--allowed-local-media-path", str(QI_WORK),
    "--middleware", "ct_qwen_image_middleware.QwenImageMiddleware",
]
if VAE_TILING:
    SERVER_ARGS.append("--vae-use-tiling")
if STEP_EXECUTION:
    SERVER_ARGS.append("--step-execution")

if QUANTIZATION == "fp8":
    SERVER_ARGS.extend([
        "--diffusion-quantization-config",
        json.dumps({"transformer": {"method": "fp8", "ignored_layers": ["img_mlp"]}}),
    ])
elif QUANTIZATION == "fp8-text":
    SERVER_ARGS.extend([
        "--diffusion-quantization-config",
        json.dumps({"text_encoder": {"method": "fp8"}}),
    ])
elif QUANTIZATION == "fp8-all":
    SERVER_ARGS.extend([
        "--diffusion-quantization-config",
        json.dumps({
            "transformer": {"method": "fp8", "ignored_layers": ["img_mlp"]},
            "text_encoder": {"method": "fp8"},
        }),
    ])

if PREFIX_KV_DTYPE in ("fp8_v", "fp8"):
    SERVER_ARGS.extend([
        "--stage-overrides",
        json.dumps({"0": {"extras": {"prefix_kv_cache_dtype": PREFIX_KV_DTYPE}}}),
    ])

if PRELOADED_LORA_LOCAL:
    SERVER_ARGS.extend(["--lora-path", PRELOADED_LORA_LOCAL, "--lora-scale", str(LORA_SCALE)])

# Reclaim GPU VRAM from any leftover workers of an earlier notebook execution on the same VM
subprocess.run(["pkill", "-f", f"{VENV_DIR}/bin/"], check=False)
time.sleep(2)
subprocess.run(["pkill", "-9", "-f", f"{VENV_DIR}/bin/|DiffusionWorker|EngineCore"], check=False)
time.sleep(1)
try:
    Path(f"{WORK}/service.log").unlink(missing_ok=True)
except OSError:
    pass

def start_service():
    if "service" in PROCS:
        ct_stop_proc(PROCS["service"][0])
    subprocess.run(["pkill", "-9", "-f", f"port {SERVICE_PORT}"], check=False)
    time.sleep(1)
    qilog(
        f"Starting vLLM-Omni Qwen-Image-2.1 server on 127.0.0.1:{SERVICE_PORT} "
        f"(model={MODEL_REPO}, quant={QUANTIZATION}, step_exec={STEP_EXECUTION})"
    )
    return ct_spawn("service", SERVER_ARGS, env=VLLM_ENV, cwd=str(QI_WORK))

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify auth, run warmup generation, archive JIT kernels & supervise
if not globals().get("QI_SETUP_OK") or "service" not in PROCS:
    raise RuntimeError("Qwen-Image-2.1 service did not start because an earlier notebook cell failed; see traceback above.")
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=30):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url, headers=headers, data=None if payload is None else json.dumps(payload).encode("utf-8")
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
            return resp.status, (json.loads(raw) if raw.strip().startswith(("{", "[")) else {})
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-token-probe", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

def _abort_startup(msg):
    for _p, _ in list(PROCS.values()):
        ct_stop_proc(_p)
    raise RuntimeError(msg)

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 2400
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        _abort_startup("vLLM-Omni Qwen-Image-2.1 server exited during startup:\n" + ct_tail("service", 80))
    try:
        code_main, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code_main == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 30:
        last_line = (ct_tail("service", 3).strip().splitlines() or ["starting..."])[-1]
        qilog(f"Waiting for Qwen-Image-2.1 init | GPU: {ct_gpu_stat()} | {last_line[:100]}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    _abort_startup("Timed out waiting for vLLM-Omni Qwen-Image-2.1 server:\n" + ct_tail("service", 80))

qilog(f"Qwen-Image-2.1 server ready on {LOCAL_BASE} | GPU: {ct_gpu_stat()}")
auth_check(LOCAL_BASE, API_TOKEN)

# Fast 2-step 512x512 warmup generation to verify DiT + Qwen3-VL-8B + 4ch RGBA VAE and populate JIT caches
try:
    w_code, w_body = http_json(
        LOCAL_BASE + "/v1/images/generations",
        API_TOKEN,
        payload={
            "model": MODEL_REPO,
            "prompt": "a ceramic teapot on a wooden table, soft window light",
            "size": "512x512",
            "n": 1,
            "response_format": "b64_json",
            "num_inference_steps": 2,
            "true_cfg_scale": 1.0,
            "seed": 42,
        },
        timeout=300,
    )
    if w_code == 200 and (w_body.get("data") or [{}])[0].get("b64_json"):
        qilog("Warmup T2I generation succeeded (512x512, 2 steps, true_cfg_scale=1.0)")
    else:
        qilog(f"Warmup T2I returned HTTP {w_code}: {str(w_body)[:160]}")
except Exception as werr:
    qilog(f"Warmup T2I warning: {werr}")

for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        save_cache(_local_c, CACHE_ARCHIVES, CACHE_KEY, qilog)
    except (OSError, ValueError) as _err:
        qilog(f"Cache archive save warning ({_local_c.name}): {_err}")

ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    qilog,
    timeout=120,
    interval=8,
)
qilog(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    qilog,
    dead_markers=("replica-0 is dead", "EngineCore encountered an issue"),
    heartbeat=lambda up: qilog(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"),
)
